# Week 4 — XGBoost Model
FaultLine: trains an XGBoost classifier on the Week 2 feature table to predict
the same target as the ETAS baseline -- whether a fault section will have a
qualifying (M4.0+) event within the next 30 days -- so the two can be compared
fairly.

**Setup required first (run once in a terminal):**
```
pip install xgboost scikit-learn
```


In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xgboost as xgb
from sklearn.metrics import (
    precision_score, recall_score, average_precision_score,
    brier_score_loss, roc_auc_score,
)

pd.set_option("display.max_columns", None)


## Configuration

Same magnitude threshold and train/validate/test split as the ETAS baseline,
so the comparison is apples-to-apples. One addition specific to supervised
learning: an **embargo** of `FORECAST_DAYS` at each split boundary. A label
for a date near the end of training depends on real events up to 30 days
later -- without the embargo, a single event near a boundary could influence
both a training label and a validation label, which is a subtle leak between
sets even though no individual *feature* looks into the future.

In [2]:
MAG_THRESHOLD = 4.0
FORECAST_DAYS = 30
MC = 2.0

TRAIN_END = pd.Timestamp("2021-12-31")
VALIDATE_START = pd.Timestamp("2022-01-01")
VALIDATE_END = pd.Timestamp("2022-12-31")
TEST_START = pd.Timestamp("2023-01-01")
TEST_END = pd.Timestamp("2025-12-31")

PROJECTED_CRS = "EPSG:3310"
MAX_ASSIGN_DIST_KM = 20


## Load Week 2 outputs

`features.parquet` has the daily per-segment inputs, but not individual event
magnitudes (seismicity rate is an aggregated count, not per-event detail) --
so building the forward-looking label requires the event-level
`quakes_assigned` table, rebuilt the same way as in the ETAS notebook.

In [3]:
features = pd.read_parquet("../data/processed/features.parquet")
features["date"] = pd.to_datetime(features["date"])
if features["date"].dt.tz is not None:
    features["date"] = features["date"].dt.tz_localize(None)

faults = gpd.read_file("../data/raw/san_andreas_faults.geojson")
faults = faults.dissolve(by="section_na").reset_index()
faults["fault_segment_id"] = faults.index.astype(str)

quakes = pd.read_parquet("../data/raw/usgs_2016_2026.parquet")
quakes["time"] = pd.to_datetime(quakes["time"]).dt.tz_localize(None)
# NOTE: confirm your real column names here -- this project's schema has
# varied between 'mag'/'magnitud' and 'magnitude'/'event_id' across sessions;
# check quakes.columns.tolist() and adjust the renames below if needed.
quakes = quakes.rename(columns={"event_id": "id", "mag": "magnitude"})

quake_points_m = gpd.GeoDataFrame(
    quakes.copy(), geometry=gpd.points_from_xy(quakes["longitude"], quakes["latitude"]), crs="EPSG:4326"
).to_crs(PROJECTED_CRS)
faults_m = faults.to_crs(PROJECTED_CRS)
joined = gpd.sjoin_nearest(
    quake_points_m, faults_m[["fault_segment_id", "geometry"]],
    max_distance=MAX_ASSIGN_DIST_KM * 1000, distance_col="dist_to_fault_m",
)
quakes_assigned = joined.drop(columns="geometry").rename(columns={"time": "event_time"})

print(features.shape)
print(f"Earthquakes assigned to the region: {len(quakes_assigned):,}")


(36520, 14)
Earthquakes assigned to the region: 12,030


## Build the forward-looking label

For each `(fault_segment_id, date)` row already in `features.parquet`: does a
qualifying event occur on that segment within `(date, date + 30 days]`? Tested
against synthetic data before being applied here -- confirmed to produce
exactly the expected number of positive labels bounding a known test event,
and to correctly reject sub-threshold magnitudes.

In [4]:
def build_labels(features_dates, quakes_assigned, mag_threshold, forecast_days):
    qualifying = quakes_assigned[quakes_assigned["magnitude"] >= mag_threshold].copy()
    qualifying["event_date"] = qualifying["event_time"].dt.floor("D")

    labels = []
    for seg_id, seg_dates in features_dates.groupby("fault_segment_id"):
        seg_events = qualifying[qualifying["fault_segment_id"] == seg_id]["event_date"].sort_values().values
        for d in seg_dates["date"]:
            window_end = d + pd.Timedelta(days=forecast_days)
            has_event = np.any((seg_events > np.datetime64(d)) & (seg_events <= np.datetime64(window_end)))
            labels.append({"fault_segment_id": seg_id, "date": d, "label": int(has_event)})
    return pd.DataFrame(labels)

labels = build_labels(features[["fault_segment_id", "date"]], quakes_assigned, MAG_THRESHOLD, FORECAST_DAYS)
data = features.merge(labels, on=["fault_segment_id", "date"], how="left")

print(f"Overall positive rate: {data['label'].mean():.4f}")
print(data.groupby("fault_segment_id")["label"].mean().sort_values(ascending=False))


Overall positive rate: 0.0179
fault_segment_id
2    0.073932
8    0.032859
7    0.031216
1    0.024918
5    0.016429
0    0.000000
4    0.000000
3    0.000000
6    0.000000
9    0.000000
Name: label, dtype: float64


**Worth checking against the ETAS baseline here:** the per-segment positive
rate above is the *historical, realized* frequency of a qualifying event in
any 30-day window -- it should roughly track the same ranking as the ETAS
30-day probabilities (Creeping highest, Shelter Cove/North Coast/Mojave near
the bottom). They won't match exactly -- ETAS is a forward-looking model
output, this is a backward-looking historical frequency -- but a wildly
different ranking here would be worth investigating before trusting either
number.

## Assign train / validate / test splits, with the embargo

Tested earlier against synthetic dates spanning a split boundary -- confirmed
a clean 30-day gap with no date landing in more than one split.

In [5]:
def assign_split(date, forecast_days, train_end, validate_start, validate_end, test_start, test_end):
    if date <= train_end - pd.Timedelta(days=forecast_days):
        return "train"
    elif validate_start <= date <= validate_end - pd.Timedelta(days=forecast_days):
        return "validate"
    elif test_start <= date <= test_end - pd.Timedelta(days=forecast_days):
        return "test"
    else:
        return "embargo"

data["split"] = data["date"].apply(
    lambda d: assign_split(d, FORECAST_DAYS, TRAIN_END, VALIDATE_START, VALIDATE_END, TEST_START, TEST_END)
)
print(data["split"].value_counts())
data = data[data["split"] != "embargo"].copy()


split
train       21610
test        10660
validate     3350
embargo       900
Name: count, dtype: int64


## Prepare features and train

`fault_segment_id` is passed as a native pandas categorical rather than
one-hot encoded -- XGBoost's `enable_categorical=True` handles this directly,
letting the model learn segment-specific base rates alongside the numeric
features, without manually exploding it into 10 separate columns.

`scale_pos_weight` is computed from the **training set only** -- using the
full dataset's class balance here would leak information about the
validate/test distribution into training.

In [ ]:
FEATURE_COLS = [
    "fault_segment_id", "seismicity_rate_7d", "seismicity_rate_30d", "seismicity_rate_90d",
    "seismicity_rate_30d_z", "b_value", "b_value_se", "n_events", "b_value_reliable",
    "gps_strain_rate", "gps_station_distance_km", "gps_reliable",
    "max_mag_7d", "max_mag_30d", "days_since_last_m35",
]

data["fault_segment_id"] = data["fault_segment_id"].astype("category")

train = data[data["split"] == "train"]
validate = data[data["split"] == "validate"]
test = data[data["split"] == "test"]

X_train, y_train = train[FEATURE_COLS], train["label"]
X_validate, y_validate = validate[FEATURE_COLS], validate["label"]
X_test, y_test = test[FEATURE_COLS], test["label"]

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"scale_pos_weight: {scale_pos_weight:.2f}")
print(f"Train: {len(train)} rows, {y_train.mean():.4f} positive rate")
print(f"Validate: {len(validate)} rows, {y_validate.mean():.4f} positive rate")
print(f"Test: {len(test)} rows, {y_test.mean():.4f} positive rate")

model = xgb.XGBClassifier(
    enable_categorical=True,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
)
model.fit(X_train, y_train, eval_set=[(X_validate, y_validate)], verbose=False)


scale_pos_weight: 53.71
Train: 21610 rows, 0.0183 positive rate
Validate: 3350 rows, 0.0179 positive rate
Test: 10660 rows, 0.0188 positive rate


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## Evaluate — using metrics that actually matter here, not raw accuracy

With a rare-event problem, a model that always predicts "no event" would
score well over 95% accuracy while being useless. Precision, recall, PR-AUC,
and calibration (Brier score) are what your proposal committed to using
instead, per the peer-feedback discussion.

In [7]:
def evaluate(model, X, y, split_name):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    print(f"--- {split_name} ---")
    print(f"Positive rate: {y.mean():.4f}")
    print(f"Precision: {precision_score(y, preds, zero_division=0):.3f}")
    print(f"Recall: {recall_score(y, preds, zero_division=0):.3f}")
    print(f"PR-AUC: {average_precision_score(y, probs):.3f}")
    print(f"ROC-AUC: {roc_auc_score(y, probs):.3f}")
    print(f"Brier score (lower is better-calibrated): {brier_score_loss(y, probs):.4f}")
    print()
    return probs

validate_probs = evaluate(model, X_validate, y_validate, "Validate")
test_probs = evaluate(model, X_test, y_test, "Test")


--- Validate ---
Positive rate: 0.0179
Precision: 0.098
Recall: 0.467
PR-AUC: 0.094
ROC-AUC: 0.873
Brier score (lower is better-calibrated): 0.0639

--- Test ---
Positive rate: 0.0188
Precision: 0.058
Recall: 0.200
PR-AUC: 0.049
ROC-AUC: 0.715
Brier score (lower is better-calibrated): 0.0532



## Feature importance — the part that speaks directly to your project's thesis

This is the ablation-adjacent check worth looking at immediately: does
`gps_strain_rate` show up as meaningfully important, or is the model
essentially ignoring it in favor of seismicity-derived features? A low
importance score here wouldn't necessarily kill your thesis (the *ablation
test* -- train with vs. without strain rate -- is the real test), but it's
worth a first look before running that more expensive comparison.

In [8]:
importance = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print(importance)


fault_segment_id           0.512244
b_value_se                 0.070334
seismicity_rate_90d        0.066855
n_events                   0.066490
gps_station_distance_km    0.065992
b_value                    0.065423
gps_strain_rate            0.049830
seismicity_rate_30d_z      0.044562
seismicity_rate_30d        0.036969
seismicity_rate_7d         0.021302
b_value_reliable           0.000000
gps_reliable               0.000000
dtype: float32


In [9]:
print(features["gps_strain_rate"].isna().mean())
print(features["gps_strain_rate"].describe())
print(X_train["gps_strain_rate"].isna().mean())

print(station_strain[["north_strain_rate", "east_strain_rate"]].describe())

0.0
count    36520.000000
mean        27.670689
std         14.642717
min          0.000000
25%         19.380615
50%         30.711634
75%         38.244993
max         66.761948
Name: gps_strain_rate, dtype: float64
0.0


NameError: name 'station_strain' is not defined

## Save predictions for later comparison against ETAS

Same shape as the ETAS baseline output (`probability_30d` per section), using
the most recent available date in the test set as the "current" forecast --
so the two models' outputs can be placed side by side.

In [ ]:
latest_date = test["date"].max()
latest_rows = test[test["date"] == latest_date].copy()
latest_rows["probability_30d"] = model.predict_proba(latest_rows[FEATURE_COLS])[:, 1]

xgboost_forecast = latest_rows.merge(
    faults[["fault_segment_id", "section_na"]], on="fault_segment_id"
)[["fault_segment_id", "section_na", "probability_30d"]].sort_values("probability_30d", ascending=False)

print(f"Forecast as of {latest_date.date()}:")
print(xgboost_forecast)

xgboost_forecast.to_parquet("../data/processed/xgboost_forecast.parquet", index=False)
import os
os.makedirs("../models", exist_ok=True)
model.save_model("../models/xgboost_model.json")
print("\nSaved forecast and model.")


Forecast as of 2025-12-01:
  fault_segment_id                        section_na  probability_30d
2                2                  Creeping section         0.270087
1                1                 Coachella section         0.009765
5                5                 Parkfield section         0.006997
7                7  San Bernardino Mountains section         0.005962
0                0           Cholame-Carrizo section         0.000511
3                3                    Mojave section         0.000511
4                4               North Coast section         0.000333
9                9              Shelter Cove Section         0.000333
8                8      Santa Cruz Mountains section         0.000188
6                6                 Peninsula section         0.000023

Saved forecast and model.
